<a href="https://colab.research.google.com/github/Saharochek98/Statistics-of-Stochastic-Processes/blob/main/Project_US_Interest_Rates_SDE_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Assignment

0. The solution must build on the original code developed during the practical laboratory sessions.

1. Plot the historical series (daily or monthly observations). Compute descriptive statistics, including the mean and variance of interest-rate changes; examine outliers and possible structural changes (change points). Test interest-rate changes for normality and provide illustrative plots.
2. Estimate the parameters of Brownian Motion (BM), constant-elasticity-of-variance (CEV), Cox–Ingersoll–Ross (CIR), geometric Brownian motion (GBM), Ornstein–Uhlenbeck (OU), and Chan–Karolyi–Longstaff–Sanders (CKLS) models by maximum likelihood, using the Euler, Kessler, and Shoji–Ozaki Gaussian transition approximations.
3. Present parameter estimates for each model and approximation in a consolidated table.
4. Calculate AIC and BIC, identify the best-fitting model, and tabulate the criteria.
5. Repeat the estimation with the [pymle library](https://github.com/jkirkby3/pymle).
6. Compare the implementations and provide illustrative plots overlaying fitted-model results and historical observations.
7. Discuss the findings in detail.

**Additional validation:** multi-start likelihood optimization, standardized-innovation diagnostics, and conditional one-step-ahead predictions. These additions strengthen the original assignment without replacing its laboratory-based approach.

### Project 3 — US Interest Rates (June 1964–December 1989; monthly observations)

Compare the fitted results with [Brouste et al., *The YUIMA Project: A Computational Framework for Simulation and Inference of Stochastic Differential Equations*](https://www.semanticscholar.org/paper/The-YUIMA-Project%3A-A-Computational-Framework-for-Brouste-Iacus). Any parameter values quoted from the literature should be treated as contextual benchmarks, not directly comparable estimates unless the data definition, sample period, scaling, and estimation method agree.

### Part 1 — Historical observations, descriptive statistics, outliers, normality, and structural changes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm
from scipy.optimize import minimize
from scipy.stats import shapiro, jarque_bera, anderson
from scipy.signal import argrelextrema

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
from statsmodels.graphics.tsaplots import plot_acf

In [ ]:
# Load and preprocess the Excel data
def load_rates_excel(filepath, start_year=1964, start_month=6, end_year=1989, end_month=12):
    """Loads the Excel data
    Parameters:
    filepath: str
        Path to the .xlsx file
    start_year, start_month: int
        Start date (June 1964)
    end_year, end_month : int
        End date (December 1989)
    Returns
      pd.Series with a datetime index and interest-rate values"""
    df_raw = pd.read_excel(filepath, sheet_name='Лист1', header=None)
    months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    data_list = []

    for idx, row in df_raw.iterrows():
        year = row.iloc[0]
        if pd.isna(year) or not isinstance(year, (int, float)):
            continue
        year = int(year)

        for m_idx, month in enumerate(months):
            value = row.iloc[m_idx + 1]
            if pd.notna(value):
                date = pd.Timestamp(year=year, month=m_idx+1, day=1)
                data_list.append({'date': date, 'rate': float(value)})

    df_long = pd.DataFrame(data_list).set_index('date').sort_index()

    mask = (df_long.index >= pd.Timestamp(f'{start_year}-{start_month}-01')) & \
           (df_long.index <= pd.Timestamp(f'{end_year}-{end_month}-01'))

    return df_long.loc[mask, 'rate']

rates = load_rates_excel('Project_3.xlsx')
print(f"Loaded {len(rates)} observations")
print(f"Period: {rates.index[0].date()} — {rates.index[-1].date()}")
print(f"Range: [{rates.min():.3f}, {rates.max():.3f}], mean: {rates.mean():.3f}")

print("Interest-rate descriptive statistics:")
print(f"Number of observations: {len(rates)}")
print(f"Mean: {np.mean(rates):.4f}")
print(f"Std: {np.std(rates, ddof=1):.4f}")
print(f"Variance: {np.var(rates, ddof=1):.6f}")
print(f"Median: {np.median(rates):.4f}")
print(f"Min/Max: [{rates.min():.4f}, {rates.max():.4f}]")

# Plot monthly historical observations
plt.plot(rates.index, rates.values, 'b-', linewidth=1.2, label='US interest rate')
plt.axhline(rates.mean(), color='red', linestyle='--', linewidth=1,
            label=f'Mean: {rates.mean():.2f}')
plt.title('US interest rates (monthly, 1964–1989)', fontsize=12, fontweight='bold')
plt.xlabel('Year'); plt.ylabel('Interest rate (%)')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Calculate changes and descriptive statistics
diff_rates = np.diff(rates.values)
dates_diff = rates.index[1:]

print("Interest-rate change statistics:")
print(f"Number of observations: {len(diff_rates)}")
print(f"Mean: {np.mean(diff_rates):.4f}")
print(f"Std: {np.std(diff_rates, ddof=1):.4f}")
print(f"Variance: {np.var(diff_rates, ddof=1):.6f}")
print(f"Median: {np.median(diff_rates):.4f}")
print(f"Min/Max: [{diff_rates.min():.4f}, {diff_rates.max():.4f}]")

# Plot rate changes
plt.plot(dates_diff, diff_rates, 'g-', linewidth=0.8, alpha=0.8)
plt.axhline(0, color='black', linestyle='-', linewidth=0.5)
plt.title('Interest-rate changes', fontsize=12)
plt.xlabel('Year'); plt.ylabel('ΔX')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Outlier analysis (IQR rule)
Q1 = np.percentile(diff_rates, 25)
Q3 = np.percentile(diff_rates, 75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_mask = (diff_rates < lower_bound) | (diff_rates > upper_bound)
outlier_indices = np.where(outliers_mask)[0]

print("Outliers in rate changes (IQR rule):")
print(f"IQR = {IQR:.4f}, bounds: [{lower_bound:.4f}, {upper_bound:.4f}]")
print(f"Outliers found: {len(outlier_indices)} out of {len(diff_rates)} ({100*len(outlier_indices)/len(diff_rates):.1f}%)")

if len(outlier_indices) > 0:
    print(f"Outlier dates:")
    for idx in outlier_indices[:10]:  # first 10
        print(f"{dates_diff[idx].date()}: {diff_rates[idx]:.4f}")

plt.plot(dates_diff, diff_rates, 'g-', linewidth=0.8, alpha=0.6, label='ΔX')
plt.scatter(dates_diff[outliers_mask], diff_rates[outliers_mask],
            color='red', s=30, zorder=5, label='Outliers (IQR)', marker='x')
plt.axhline(lower_bound, color='orange', linestyle='--', linewidth=1, alpha=0.7, label='IQR bounds')
plt.axhline(upper_bound, color='orange', linestyle='--', linewidth=1, alpha=0.7)
plt.axhline(0, color='black', linestyle='-', linewidth=0.5)
plt.title('Outliers in interest-rate changes', fontsize=12)
plt.xlabel('Year'); plt.ylabel('ΔX')
plt.legend(); plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Test rate changes for normality
# Shapiro-Wilk
sw_stat, sw_p = shapiro(diff_rates)
print(f"Shapiro-Wilk:  W = {sw_stat:.4f}, p-value = {sw_p:.4f} → {'fail to reject normality' if sw_p > 0.05 else 'reject normality'}")

# Jarque-Bera
jb_stat, jb_p = jarque_bera(diff_rates)
print(f"Jarque-Bera:   JB = {jb_stat:.3f}, p-value = {jb_p:.4f} → {'fail to reject normality' if jb_p > 0.05 else 'reject normality'}")

# Anderson-Darling
ad_result = anderson(diff_rates, dist='norm')
print(f"Anderson-Darling: A² = {ad_result.statistic:.3f}")
print(f"Critical value (5%): {ad_result.critical_values[2]:.3f} → {'fail to reject normality' if ad_result.statistic < ad_result.critical_values[2] else 'reject normality'}")

# Histogram and Q–Q plot
plt.subplot(1, 2, 1)
plt.hist(diff_rates, bins=25, density=True, alpha=0.7, edgecolor='black', label='ΔX')
x_norm = np.linspace(diff_rates.min(), diff_rates.max(), 100)
plt.plot(x_norm, stats.norm.pdf(x_norm, np.mean(diff_rates), np.std(diff_rates)),
         'r-', linewidth=2, label='Normal density')
plt.title('Rate changes and fitted normal density')
plt.xlabel('ΔX, percentage points'); plt.ylabel('Density')
plt.legend(); plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
stats.probplot(diff_rates, dist="norm", plot=plt)
plt.title('Normal Q–Q plot')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('\n')
# Autocorrelation function
plot_acf(diff_rates, lags=24, alpha=0.05)
plt.title('Autocorrelation function of interest-rate changes')
plt.xlabel('Lag (months)')
plt.tight_layout()
plt.show()

In [ ]:
expected_dates = pd.date_range(
    start="1964-06-01",
    end="1989-12-01",
    freq="MS"
)

missing_dates = expected_dates.difference(rates.index)

print("Expected observations:", len(expected_dates))
print("Actual observations:", len(rates))
print("Missing months:", len(missing_dates))

if len(missing_dates):
    print("Missing dates:", missing_dates.tolist())

assert rates.index.is_unique, "Duplicate dates detected"
assert len(missing_dates) == 0, "Missing monthly observations detected"

In [ ]:
def moving_block_resample(data, rng, block_length):
    """Circular moving-block resampling preserves short-range serial dependence."""
    n = len(data)
    starts = rng.integers(0,n,size=int(np.ceil(n/block_length)))
    indices = np.concatenate([(start+np.arange(block_length))%n for start in starts])[:n]
    return data[indices]

def find_change_points(data, significance_level=0.05, min_segment_length=12):
    rng = np.random.default_rng(2026)
    n = len(data)

    mean_val = np.mean(data)
    deviations = data - mean_val
    cumulative_sum = np.cumsum(deviations)

    sigma = np.std(data, ddof=1)
    D_k = np.abs(cumulative_sum) / (sigma * np.sqrt(n))

    def find_all_change_points(segment_data, segment_start_idx):
        if len(segment_data) < 2 * min_segment_length:
            return []

        segment_n = len(segment_data)
        segment_mean = np.mean(segment_data)
        segment_deviations = segment_data - segment_mean
        segment_cumsum = np.cumsum(segment_deviations)
        segment_sigma = np.std(segment_data, ddof=1)

        if segment_sigma == 0:
            return []

        segment_D = np.abs(segment_cumsum) / (segment_sigma * np.sqrt(segment_n))
        # Restrict candidates to splits with sufficiently long left/right segments.
        segment_D[:min_segment_length-1] = -np.inf
        segment_D[segment_n-min_segment_length:] = -np.inf
        max_D_idx = np.argmax(segment_D)
        max_D = segment_D[max_D_idx]

        n_bootstrap = 1000
        bootstrap_max = np.zeros(n_bootstrap)

        for b in range(n_bootstrap):
            bootstrap_sample = moving_block_resample(segment_data, rng, block_length=max(3, int(np.sqrt(segment_n))))
            bootstrap_mean = np.mean(bootstrap_sample)
            bootstrap_dev = bootstrap_sample - bootstrap_mean
            bootstrap_cumsum = np.cumsum(bootstrap_dev)
            bootstrap_sigma = np.std(bootstrap_sample, ddof=1)
            if bootstrap_sigma > 0:
                bootstrap_D = np.abs(bootstrap_cumsum) / (bootstrap_sigma * np.sqrt(segment_n))
                bootstrap_D[:min_segment_length-1] = -np.inf
                bootstrap_D[segment_n-min_segment_length:] = -np.inf
                bootstrap_max[b] = np.max(bootstrap_D)

        p_value = (1 + np.sum(bootstrap_max >= max_D)) / (n_bootstrap + 1)

        if p_value < significance_level:
            change_idx = max_D_idx + 1

            left_changes = find_all_change_points(segment_data[:change_idx],
                                                  segment_start_idx)
            right_changes = find_all_change_points(segment_data[change_idx:],
                                                   segment_start_idx + change_idx)

            return left_changes + [segment_start_idx + change_idx] + right_changes
        else:
            return []

    change_indices = find_all_change_points(data, 0)
    change_indices = sorted(set(change_indices))

    return change_indices, D_k

rates_values = rates.values
rates_dates = rates.index
change_points, D_k_stat = find_change_points(rates_values,
                                                   significance_level=0.05,
                                                   min_segment_length=12)

print(f"Detected change points: {len(change_points)}")
for cp in change_points:
    print(f"  {rates_dates[cp].date()}: value = {rates_values[cp]:.3f}")

fig, axes = plt.subplots(2, 1, figsize=(12, 10))

axes[0].plot(rates_dates, rates_values, 'b-', linewidth=1.2, label='Interest rate')
for cp in change_points:
    axes[0].axvline(x=rates_dates[cp], color='red', linestyle='--',
                    alpha=0.7, linewidth=1)
    axes[0].plot(rates_dates[cp], rates_values[cp], 'ro', markersize=8,
                 label='Change point' if cp == change_points[0] else '')
axes[0].set_title('US interest rates with candidate change points',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Interest rate (%)')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

colors = plt.cm.tab10(np.linspace(0, 1, len(change_points) + 1))
segment_boundaries = [0] + change_points + [len(rates_values)]
for i in range(len(segment_boundaries) - 1):
    start_idx = segment_boundaries[i]
    end_idx = segment_boundaries[i + 1]
    axes[1].plot(rates_dates[start_idx:end_idx],
                 rates_values[start_idx:end_idx],
                 color=colors[i], linewidth=1.5,
                 label=f'Regime {i+1}' if i == 0 else f'Regime {i+1}')

for cp in change_points:
    axes[1].axvline(x=rates_dates[cp], color='black', linestyle='--',
                    alpha=0.5, linewidth=1)

axes[1].set_title('Candidate interest-rate regimes', fontsize=12)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Interest rate (%)')
axes[1].legend(loc='upper left', ncol=min(len(change_points) + 1, 4))
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


**Change-point caveat.** The circular moving-block bootstrap approximately preserves short-range dependence, unlike random permutation. Recursive change-point detection remains exploratory: the bootstrap null, block length, and multiple testing affect the inferred number of breaks.

### Part 2 — Maximum likelihood with Euler, Kessler, and Shoji–Ozaki approximations

In [ ]:
# SDE model classes
class SDEModel:
    """Base class for models dX = b(X)dt + σ(X)dW"""
    def __init__(self, params, param_names, bounds=None):
        self.params = np.array(params)
        self.param_names = param_names
        self.bounds = bounds

    def drift(self, x): raise NotImplementedError
    def diffusion(self, x): raise NotImplementedError
    def n_params(self): return len(self.params)

class BrownianMotion(SDEModel):
    """dX = μ·dt + σ·dW"""
    def __init__(self, mu=0.0, sigma=0.1):
        super().__init__([mu, sigma], ['μ', 'σ'], bounds=[(-10, 10), (1e-4, 10)])
    def drift(self, x): return np.full_like(x, self.params[0])
    def diffusion(self, x): return np.full_like(x, self.params[1])

class GBM(SDEModel):
    """dX = μ·X·dt + σ·X·dW"""
    def __init__(self, mu=0.05, sigma=0.2):
        super().__init__([mu, sigma], ['μ', 'σ'], bounds=[(-1, 1), (1e-4, 2)])
    def drift(self, x): return self.params[0] * x
    def diffusion(self, x): return self.params[1] * np.abs(x)

class OU(SDEModel):
    """dX = κ(θ - X)dt + σ·dW"""
    def __init__(self, kappa=1.0, theta=5.0, sigma=0.5):
        super().__init__([kappa, theta, sigma], ['κ', 'θ', 'σ'],
                        bounds=[(1e-3, 10), (0, 20), (1e-4, 5)])
    def drift(self, x): return self.params[0] * (self.params[1] - x)
    def diffusion(self, x): return np.full_like(x, self.params[2])

class CIR(SDEModel):
    """dX = κ(θ - X)dt + σ·√X·dW"""
    def __init__(self, kappa=1.0, theta=5.0, sigma=0.5):
        super().__init__([kappa, theta, sigma], ['κ', 'θ', 'σ'],
                        bounds=[(1e-3, 10), (0.1, 20), (1e-4, 3)])
    def drift(self, x): return self.params[0] * (self.params[1] - np.maximum(x, 0))
    def diffusion(self, x): return self.params[2] * np.sqrt(np.maximum(x, 1e-8))

class CEV(SDEModel):
    """dX = κ(θ - X)dt + σ·|X|^γ·dW"""
    def __init__(self, kappa=1.0, theta=5.0, sigma=0.3, gamma=0.5):
        super().__init__([kappa, theta, sigma, gamma], ['κ', 'θ', 'σ', 'γ'],
                        bounds=[(1e-3, 10), (0.1, 20), (1e-4, 3), (0, 2)])
    def drift(self, x): return self.params[0] * (self.params[1] - x)
    def diffusion(self, x): return self.params[2] * np.power(np.abs(x), self.params[3])

class CKLS(SDEModel):
    """dX = (α + β·X)dt + σ·|X|^γ·dW"""
    def __init__(self, alpha=1.0, beta=-0.1, sigma=0.3, gamma=0.5):
        super().__init__([alpha, beta, sigma, gamma], ['α', 'β', 'σ', 'γ'],
                        bounds=[(-5, 10), (-2, 0.5), (1e-4, 3), (0, 2)])
    def drift(self, x): return self.params[0] + self.params[1] * x
    def diffusion(self, x): return self.params[2] * np.power(np.abs(x), self.params[3])


# Gaussian transition approximations, based on the laboratory model classes.
# All six drift functions are affine: b(x)=c+L*x, with b''(x)=0.

def transition_moments(x, dt, model, method):
    x = np.asarray(x, dtype=float)
    b = np.asarray(model.drift(x), dtype=float)
    sig = np.asarray(model.diffusion(x), dtype=float)
    a = sig**2
    if np.any(~np.isfinite(a)) or np.any(a <= 0):
        raise ValueError("Diffusion variance must be positive and finite")
    if isinstance(model, (BrownianMotion, GBM)):
        L = 0.0 if isinstance(model, BrownianMotion) else float(model.params[0])
    elif isinstance(model, CKLS):
        L = float(model.params[1])
    else:
        L = -float(model.params[0])
    if method == 'euler':
        return x + b*dt, a*dt
    if method == 'shoji_ozaki':
        # Stable phi-functions; locally freeze diffusion at the previous state.
        z = L*dt
        phi1 = np.expm1(z)/z if abs(z) > 1e-8 else 1 + z/2 + z*z/6
        phi2 = np.expm1(2*z)/(2*z) if abs(z) > 1e-8 else 1 + z + 2*z*z/3
        return x + b*dt*phi1, a*dt*phi2
    if method == 'kessler':
        # Second-order conditional moments from the diffusion generator.
        # L(b)=b*b'(x), because b''(x)=0.
        mean = x + b*dt + 0.5*b*L*dt**2
        # Var(X_next|x) = a*dt + (a*b' + b*a' + 0.25*a*a'')*dt^2.
        if isinstance(model, (BrownianMotion, OU)):
            ap = np.zeros_like(x); app = np.zeros_like(x)
        else:
            if isinstance(model, CIR):
                gamma = 0.5
            elif isinstance(model, GBM):
                gamma = 1.0
            else:
                gamma = float(model.params[-1])
            # Observed rates are strictly positive; power derivatives use x>0.
            xp = np.maximum(x, 1e-10)
            ap = 2*gamma*a/xp
            app = 2*gamma*(2*gamma-1)*a/xp**2
        variance = a*dt + (a*L + b*ap + 0.25*a*app)*dt**2
        if np.any(~np.isfinite(variance)) or np.any(variance <= 0):
            raise ValueError("Invalid Kessler variance")
        return mean, variance
    raise ValueError(f"Unknown approximation: {method}")


def gaussian_loglik(x_prev, x_next, dt, model, method):
    mean, variance = transition_moments(x_prev, dt, model, method)
    return norm.logpdf(x_next, loc=mean, scale=np.sqrt(variance))

def euler_loglik(x_prev, x_next, dt, model):
    return gaussian_loglik(x_prev, x_next, dt, model, 'euler')

def kessler_loglik(x_prev, x_next, dt, model):
    return gaussian_loglik(x_prev, x_next, dt, model, 'kessler')

def shoji_ozaki_loglik(x_prev, x_next, dt, model):
    return gaussian_loglik(x_prev, x_next, dt, model, 'shoji_ozaki')

DISCRETIZATION = {'euler': euler_loglik, 'kessler': kessler_loglik,
                  'shoji_ozaki': shoji_ozaki_loglik}

def negative_log_likelihood(params, model, x_prev, x_next, dt, loglik_fn):
    model.params = np.asarray(params, dtype=float)
    try:
        ll = loglik_fn(x_prev, x_next, dt, model)
        return -float(np.sum(ll)) if np.all(np.isfinite(ll)) else 1e12
    except (ValueError, FloatingPointError, OverflowError, ZeroDivisionError):
        return 1e12

def fit_model_mle(model, x_data, dt, method='euler', max_iter=1000, n_starts=5, seed=2026):
    x_prev, x_next = x_data[:-1], x_data[1:]
    rng = np.random.default_rng(seed)
    bounds = np.asarray(model.bounds, dtype=float)
    starts = [np.asarray(model.params, dtype=float)]
    for _ in range(n_starts-1):
        perturbation = rng.normal(0, 0.15, len(starts[0])) * (bounds[:,1]-bounds[:,0])
        starts.append(np.clip(starts[0]+perturbation, bounds[:,0]+1e-7, bounds[:,1]-1e-7))
    fits = []
    for initial in starts:
        result = minimize(negative_log_likelihood, initial,
                          args=(model, x_prev, x_next, dt, DISCRETIZATION[method]),
                          method='L-BFGS-B', bounds=model.bounds,
                          options={'maxiter': max_iter, 'ftol': 1e-10})
        if result.success and np.isfinite(result.fun) and result.fun < 1e11:
            fits.append(result)
    if not fits:
        return model, np.nan, 0, False
    best = min(fits, key=lambda r: r.fun)
    model.params = best.x.copy()
    return model, -best.fun, best.nit, True

x_data = rates.to_numpy(dtype=float)
dt = 1/12
models = {
    'BM': BrownianMotion(0.01, 0.5), 'GBM': GBM(0.01, 0.1),
    'OU': OU(1.0, 6.0, 0.5), 'CIR': CIR(1.0, 6.0, 0.5),
    'CEV': CEV(1.0, 6.0, 0.3, 0.5), 'CKLS': CKLS(1.0, -0.1, 0.3, 0.5)
}
methods = ['euler', 'kessler', 'shoji_ozaki']
results = []
for model_name, model_init in models.items():
    for method in methods:
        model = type(model_init)(*model_init.params)
        fitted, loglik, n_iter, success = fit_model_mle(model, x_data, dt, method)
        k, n_obs = fitted.n_params(), len(x_data)-1
        results.append({'Model': model_name, 'Method': method,
                        'Success': success, 'LogLik': loglik,
                        'AIC': 2*k-2*loglik if success else np.nan,
                        'BIC': k*np.log(n_obs)-2*loglik if success else np.nan,
                        'N_iter': n_iter,
                        **{name: val for name, val in zip(fitted.param_names, fitted.params)}})
df_results = pd.DataFrame(results).sort_values(['Model','Method']).reset_index(drop=True)
print(df_results.to_string(index=False))
if df_results['AIC'].notna().sum() == 0:
    raise RuntimeError('All custom MLE fits failed; check the input data and model bounds.')
print('Best custom AIC:', df_results.loc[df_results.AIC.idxmin(), ['Model','Method','AIC']].to_dict())
print('Best custom BIC:', df_results.loc[df_results.BIC.idxmin(), ['Model','Method','BIC']].to_dict())
df_results.to_csv('custom_mle_results.csv', index=False)


In [ ]:
df_results

### Part 3 — Independent estimation with the `pymle` library

In [ ]:
%pip install -q git+https://github.com/jkirkby3/pymle.git

In [ ]:
from pymle.models import BrownianMotion as PyBrownianMotion
from pymle.models import GeometricBM as PyGeometricBM
from pymle.models import OrnsteinUhlenbeck as PyOrnsteinUhlenbeck
from pymle.models import CIR as PyCIR
from pymle.models import CEV as PyCEV
from pymle.models import CKLS as PyCKLS
from pymle.core.TransitionDensity import KesslerDensity, ShojiOzakiDensity, EulerDensity
from pymle.fit.AnalyticalMLE import AnalyticalMLE


In [ ]:
sample = rates.to_numpy(dtype=float)
dt = 1/12
MODELS = {
    'BM': {'cls': PyBrownianMotion, 'names':['mu','sigma']},
    'GBM': {'cls': PyGeometricBM, 'names':['mu','sigma']},
    'OU': {'cls': PyOrnsteinUhlenbeck, 'names':['kappa','theta','sigma']},
    'CIR': {'cls': PyCIR, 'names':['kappa','theta','sigma']},
    'CEV': {'cls': PyCEV, 'names':['kappa','theta','sigma','gamma']},
    'CKLS': {'cls': PyCKLS, 'names':['alpha','beta','sigma','gamma']}
}
DENSITY = {'euler': EulerDensity, 'kessler': KesslerDensity,
           'shoji_ozaki': ShojiOzakiDensity}
pymle_results = []
for name, cfg in MODELS.items():
    init = models[name]
    # Match the custom implementation's parameter bounds and initial guess.
    for method, density_cls in DENSITY.items():
        try:
            library_model = cfg['cls']()
            fit = AnalyticalMLE(sample, init.bounds, dt,
                                density=density_cls(library_model)).estimate_params(init.params.copy())
            ll = float(fit.log_like)
            valid = np.isfinite(ll) and np.all(np.isfinite(fit.params))
            k, n_obs = len(init.params), len(sample)-1
            row = {'Model':name, 'Method':method, 'Success':bool(valid),
                   'Log_Likelihood':ll if valid else np.nan,
                   'AIC':2*k-2*ll if valid else np.nan,
                   'BIC':k*np.log(n_obs)-2*ll if valid else np.nan}
            row.update(dict(zip(cfg['names'],fit.params)))
        except Exception as exc:
            print(f'pymle failed for {name}/{method}: {type(exc).__name__}: {exc}')
            row = {'Model':name,'Method':method,'Success':False,
                   'Log_Likelihood':np.nan,'AIC':np.nan,'BIC':np.nan}
        pymle_results.append(row)
df_pymle = pd.DataFrame(pymle_results)
print(df_pymle.round(5).to_string(index=False))
df_pymle.to_csv('pymle_results.csv',index=False)


In [ ]:
df_pymle

### Part 4 — Model comparison and visualization

In [ ]:
df_pymle['AIC_rank'] = df_pymle['AIC'].rank(method='min')
df_pymle['BIC_rank'] = df_pymle['BIC'].rank(method='min')

best_by_aic = df_pymle[df_pymle['AIC_rank'] == 1.0]
best_by_bic = df_pymle[df_pymle['BIC_rank'] == 1.0]

print("Best models by AIC:")
print(best_by_aic[['Model', 'Method', 'AIC', 'BIC']].to_string(index=False))

print("\nBest models by BIC:")
print(best_by_bic[['Model', 'Method', 'AIC', 'BIC']].to_string(index=False))

In [ ]:
# Matched AIC/BIC comparison: compare identical models and approximations.
sample = rates.to_numpy(dtype=float)
dates = rates.index
dt = 1/12
comparison = df_results.merge(df_pymle, on=['Model','Method'], suffixes=('_custom','_pymle'))
comparison['Delta_AIC'] = comparison['AIC_custom'] - comparison['AIC_pymle']
comparison['Delta_BIC'] = comparison['BIC_custom'] - comparison['BIC_pymle']
print('Matched custom / pymle comparison:')
print(comparison[['Model','Method','AIC_custom','AIC_pymle','Delta_AIC',
                  'BIC_custom','BIC_pymle','Delta_BIC']].round(3).to_string(index=False))

def get_param(row, *names):
    for name in names:
        if name in row.index and pd.notna(row[name]):
            return float(row[name])
    return np.nan

def extract_ckls_params(row):
    name = row['Model']
    sigma = get_param(row,'sigma','σ')
    if name == 'CKLS':
        return get_param(row,'alpha','α'), get_param(row,'beta','β'), sigma, get_param(row,'gamma','γ')
    if name in ('CEV','CIR','OU'):
        kappa, theta = get_param(row,'kappa','κ'), get_param(row,'theta','θ')
        gamma = {'CIR':0.5,'OU':0.0}.get(name,get_param(row,'gamma','γ'))
        return kappa*theta, -kappa, sigma, gamma
    if name == 'GBM':
        return 0.0, get_param(row,'mu','μ'), sigma, 1.0
    if name == 'BM':
        return get_param(row,'mu','μ'), 0.0, sigma, 0.0
    raise ValueError(name)

def simulate_ckls(alpha,beta,sigma,gamma,x0,n_steps,dt,n_paths=200,seed=42):
    rng = np.random.default_rng(seed)
    paths = np.empty((n_steps+1,n_paths))
    paths[0] = x0
    for t in range(n_steps):
        x = paths[t]
        x_next = x + (alpha+beta*x)*dt + sigma*np.maximum(x,0)**gamma*rng.normal(0,np.sqrt(dt),n_paths)
        # Full truncation keeps level-dependent diffusion real-valued for positive-rate models.
        paths[t+1] = np.maximum(x_next, 0.0) if gamma > 0 else x_next
    return paths

article_params = {'alpha':2.08,'beta':-0.26,'sigma':0.13,'gamma':1.44}
valid_custom = df_results.dropna(subset=['AIC'])
valid_pymle = df_pymle.dropna(subset=['AIC'])
best_orig = valid_custom.loc[valid_custom.AIC.idxmin()]
best_lib = valid_pymle.loc[valid_pymle.AIC.idxmin()] if len(valid_pymle) else None
print('Custom best:', best_orig[['Model','Method','AIC','BIC']].to_dict())
if best_lib is not None:
    print('pymle best:', best_lib[['Model','Method','AIC','BIC']].to_dict())

fig, ax = plt.subplots(figsize=(13,6))
ax.plot(dates,sample,color='black',lw=1.3,label='Historical rates')
for row,label,color in [(best_orig,'Custom best','tab:blue'),
                         (best_lib,'pymle best','tab:orange')]:
    if row is None: continue
    params = extract_ckls_params(row)
    paths = simulate_ckls(*params, sample[0],len(sample)-1,dt,seed=42)
    ax.plot(dates,paths.mean(axis=1),lw=1.4,ls='--',color=color,
            label=f'{label}: {row["Model"]} / {row["Method"]}')
    ax.fill_between(dates,*np.percentile(paths,[5,95],axis=1),color=color,alpha=.10)
article_paths = simulate_ckls(**article_params,x0=sample[0],n_steps=len(sample)-1,dt=dt)
ax.plot(dates,article_paths.mean(axis=1),':',color='tab:green',
        label='Literature CKLS parameters (illustrative)')
ax.set(title='Unconditional simulation ensembles vs observed interest rates',
       xlabel='Year',ylabel='Interest rate (%)')
ax.legend();ax.grid(alpha=.25);plt.tight_layout();plt.show()
print('Note: unconditional simulation means are NOT one-step forecasts.')

# CKLS comparison across all three methods (original assignment retained).
fig,axes=plt.subplots(1,3,figsize=(17,4.5),sharey=True)
ckls_table=[]
for ax,method in zip(axes,methods):
    ax.plot(dates,sample,color='black',lw=1,label='Observed')
    for frame,label,color in [(df_results,'Custom','tab:blue'),
                              (df_pymle,'pymle','tab:orange')]:
        selected=frame[(frame.Model=='CKLS')&(frame.Method==method)].dropna(subset=['AIC'])
        if selected.empty: continue
        row=selected.iloc[0]
        pars=extract_ckls_params(row)
        paths=simulate_ckls(*pars,sample[0],len(sample)-1,dt,n_paths=100,seed=42)
        ax.plot(dates,paths.mean(axis=1),ls='--',lw=1.2,color=color,label=label)
        ckls_table.append({'Source':label,'Method':method,'alpha':pars[0],
                           'beta':pars[1],'sigma':pars[2],'gamma':pars[3],
                           'AIC':row['AIC']})
    ax.set_title(method);ax.grid(alpha=.25);ax.legend()
plt.suptitle('CKLS: custom vs pymle (unconditional simulation means)')
plt.tight_layout();plt.show()
df_ckls=pd.DataFrame(ckls_table)
print(df_ckls.round(5).to_string(index=False))


### Part 5 — Added validation: conditional forecasts and standardized innovations

For the selected model, each one-step forecast conditions on the **observed previous rate**. The standardized innovation is \(z_t=(X_t-\hat m_t)/\sqrt{\hat v_t}\), using the fitted transition moments. This is distinct from the unconditional simulation overlays above. ACF, Q–Q, and histogram diagnostics assess whether the Gaussian approximation captures residual dependence and tail behavior. Forecast errors are **in-sample conditional errors**, not out-of-sample performance.

In [ ]:
from scipy.stats import probplot
from statsmodels.graphics.tsaplots import plot_acf

def model_from_row(row):
    cls = {'BM':BrownianMotion,'GBM':GBM,'OU':OU,'CIR':CIR,
           'CEV':CEV,'CKLS':CKLS}[row['Model']]
    param_names = {'BM':['μ','σ'],'GBM':['μ','σ'],
                   'OU':['κ','θ','σ'],'CIR':['κ','θ','σ'],
                   'CEV':['κ','θ','σ','γ'],'CKLS':['α','β','σ','γ']}[row['Model']]
    return cls(*[float(row[name]) for name in param_names])

best_row = df_results.dropna(subset=['AIC']).sort_values('AIC').iloc[0]
selected_model = model_from_row(best_row)
mean_pred,var_pred = transition_moments(sample[:-1],dt,selected_model,best_row['Method'])
innovation = (sample[1:]-mean_pred)/np.sqrt(var_pred)
errors = sample[1:]-mean_pred
rmse = float(np.sqrt(np.mean(errors**2)))
mae = float(np.mean(np.abs(errors)))
print(f'Best custom model: {best_row.Model} / {best_row.Method}')
print(f'Conditional one-step RMSE: {rmse:.5f}; MAE: {mae:.5f}')
print(f'Standardized innovation mean: {innovation.mean():.4f}; '
      f'standard deviation: {innovation.std(ddof=1):.4f}')

fig,axs=plt.subplots(2,2,figsize=(13,9))
axs[0,0].plot(dates[1:],sample[1:],color='black',lw=1,label='Observed')
axs[0,0].plot(dates[1:],mean_pred,color='tab:blue',lw=1,label='Conditional mean')
axs[0,0].fill_between(dates[1:],mean_pred-1.96*np.sqrt(var_pred),
                       mean_pred+1.96*np.sqrt(var_pred),alpha=.15,
                       label='Approx. 95% conditional interval')
axs[0,0].set_title('One-step conditional predictions (in sample)');axs[0,0].legend()
axs[0,1].hist(innovation,bins=25,density=True,alpha=.6,label='Innovations')
zgrid=np.linspace(-4,4,200)
axs[0,1].plot(zgrid,norm.pdf(zgrid),'k--',label='Standard normal')
axs[0,1].set_title('Standardized innovation distribution');axs[0,1].legend()
probplot(innovation,dist='norm',plot=axs[1,0]);axs[1,0].set_title('Innovation Q–Q plot')
plot_acf(innovation,lags=min(24,len(innovation)//4),ax=axs[1,1]);axs[1,1].set_title('Innovation autocorrelation')
for ax in axs.flat:ax.grid(alpha=.2)
plt.tight_layout();plt.show()


### Conclusions and interpretation

**How to read the results.** Use the freshly computed tables above for model rankings, AIC/BIC differences, and parameter estimates; the numerical results can change after correcting the approximations and optimizer. AIC and BIC rank fitted likelihood models but do not establish predictive accuracy. Compare the same model and approximation across the custom implementation and `pymle` using consistent parameter conventions and bounds. Different transition-density implementations can still yield differences.

**Economic interpretation.** A negative CKLS slope parameter β indicates mean reversion toward −α/β when α > 0. A positive elasticity γ indicates level-dependent diffusion; γ > 1 means the *diffusion coefficient* increases more than proportionally with the level, not that variance is literally unbounded in the observed sample. Historical changes in US monetary policy can motivate structural-break checks, but detected breaks are exploratory rather than causal evidence.

**Statistical limitations.** Tests on raw monthly rate changes are not tests on standardized model innovations. Structural-change inference must account for serial dependence. In-sample fit, residual diagnostics, and conditional one-step predictions answer different questions. Report optimizer failures and boundary estimates explicitly; do not treat them as valid unconstrained MLEs. Literature estimates may differ because of sample period, data scaling, regularization, or estimation method.

**Overall conclusion.** Interpret the preferred specification using the updated AIC/BIC rankings together with innovation diagnostics and conditional forecast errors. The most flexible model need not be the most robust or best predictive model.